# Baseline models
#### Previous notebook: [Data provisioning](02_data_provisioning.ipynb)

#### Models implemented in this notebook:
 - [Logistic Regression](#logistic-regression)
 - [Linear SVC](#linear-svc)

## Load train/val/test sets
In this step, I will load the files from the split_data folder to ensure that every model is trained, tuned and evaluated on exactly the same input texts and labels. To load the data, I will use the "load_split_data" function from the `helper.py` file:

In [1]:
from utilities import helper

X_train, X_val, X_test, y_train, y_val, y_test = helper.load_split_data(show = True)
labels = list(y_train.columns)

X_train: (18243,)
X_val: (3909,)
X_test: (3910,)
y_train: (18243, 5)
y_val: (3909, 5)
y_test: (3910, 5)


## Setting TF-IDF
In this step, I create a TF-IDF vectorizer configuration function to create a new vectorizer for each model. The TF-IDF vectorizer is used to measure how important a token is - tokens that are frequent in a message but not that frequent across the whole dataset, will be assigned with higher weights. Before tokenization and weighting, the text message is normalized by the `preprocess_text` function to ensure the consistency of the input format and reduce noise - the text is lowercased; URLs, emails, user mentions, hashtags and numbers are replaced with fixed placeholder tokens; extra white spaces are removed and spaCy lemmatization is applied to reduce the words to their base or dictionary form. After that, the text is tokenized by splitting on white spaces. The *ngram_range* uses both unigrams(single words) and bigrams(pair of 2 words) to capture short phrases, which might carry important information. Lasty, *min_df* and *max_df* are used to exclude very rare and too common tokens in order to reduce noise and improve generalization.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from utilities.text_normalization import preprocess_text

def make_tfidf():
    return TfidfVectorizer(
        preprocessor=preprocess_text,
        tokenizer=str.split,
        token_pattern=None,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95
    )

## Get model scores
Different models score their predictions in different ways - some give probabilities in the range 0-1, while others base their decisions on negative and positive scores. In this notebook I implement both types of models, which is why the function below checks which scoring method the model supports and returns the correct score matrix. These scores are later compared to a chosen threshold and converted into the final binary multi-label predictions:

In [3]:
def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        return scores

## Choose best score
In this step, I create a function to select the best threshold for turning the models' scores into final 0/1 predictions. The baseline models do not output direct binary labels - instead they output confidence scores. The function takes the real labels, the confidence scores and a list of thresholds. For each threshold, it converts the confidence scores into 0/1 predictions and calculates the micro F1-score (summarizes performance across all labels). Finally, the function returns the threshold with the highest micro F1-score.

In [4]:
def best_threshold_micro_f1_from_scores(y_true, y_scores, thresholds):
    best_t, best_f1 = None, -1.0
    for t in thresholds:
        y_pred = (y_scores >= t).astype(int)
        f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

## General tuning
The purpose of this function is to construct a model, train it, test different hyperparameter values and, for each value, find the best decision threshold, using the previous function (*best_threshold_micro_f1_from_scores*). For each *param_grid* setting, it finds and saves the best threshold and micro F1-score. In the end, the function returns the parameter setting that achieved the highest micro F1-score, as well as a table presenting the results of all tested hyperparameters settings:

In [5]:
import pandas as pd

def general_tuning(X_train, y_train, X_val, y_val,make_pipeline_fn,param_grid,thresholds):
    rows = []

    for params in param_grid:
        model = make_pipeline_fn(**params)
        model.fit(X_train, y_train)

        y_scores = get_scores(model, X_val)
        best_t, best_f1 = best_threshold_micro_f1_from_scores(y_val, y_scores, thresholds)

        rows.append({
            **params,
            "threshold": float(best_t),
            "val_micro_f1": float(best_f1),
            "model": model
        })

    best = max(rows, key=lambda r: r["val_micro_f1"])
    return best, pd.DataFrame([{k: v for k, v in r.items() if k != "model"} for r in rows])

## Evaluation
The purpose of the *evalute_model* function is to evaluate an already trained model. First, it gets the model's confidence scores and uses the selected threshold to convert them into binary predictions. After that it calculates the model's micro and macro F1-score, the Hamming loss and the micro Jaccard score, and saves them along with the model's name and threshold. Finally, it returns the results with the predicted labels:

In [6]:
from sklearn.metrics import f1_score, hamming_loss, jaccard_score

def evaluate_model(model, X, y, threshold, model_name):
    y_scores = get_scores(model, X)
    y_pred = (y_scores >= threshold).astype(int)

    metrics_row = {
        "model": model_name,
        "threshold": float(threshold),
        "micro_f1": float(f1_score(y, y_pred, average="micro", zero_division=0)),
        "macro_f1": float(f1_score(y, y_pred, average="macro", zero_division=0)),
        "hamming_loss": float(hamming_loss(y, y_pred)),
        "jaccard_micro": float(jaccard_score(y, y_pred, average="micro", zero_division=0)),
    }

    return metrics_row, y_pred

The evaluation metrics used in this project are specifically chosen for proper multi-label classification evaluation:
 - **Micro F1-score** - shows the overall performance of a model across all labels; works well with imbalanced dataset; could hide poor performance on rare labels
 - **Macro F1-score** - calculates the F1-score for all labels and averages the results, gives equal importance to each label, useful to show how a model performs on rare labels
 - **Hamming loss** - shows the fraction of incorrectly predicted labels, lower results show better performance
 - **Jaccard similarity** - measures the overlap between true and predicted labels, strict - predicting extra labels or missing true labels reduces the score, higher results show better performance

# Modeling
## Logistic Regression
In this step, I will create a function that returns a new unfitted model on each run, which makes hyperparameter tuning easier. First, I create a pipeline to ensure no data leakage is introduced. The first step in the pipeline is the TF-IDF vectorizer, which converts each input message into numerical features and ensures text normalization and consistency. Then, the model is wrapped into "OneVsRestClassifier", which is very important for multi-label classification problems - creates and trains separate Logistic Regression per label and combines the outputs for the final prediction. The hyperparameter C is used to control th regularization of the model - smaller values apply stronger regularization, bigger values allow the model to fit more closely on the data. The *class_weight* is set to balanced to help the model handle label imbalance - gives more weight to the minority class during training.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline

def make_lr_pipe(C):
    return Pipeline([
    ("tfidf", make_tfidf()),
    ("clf", OneVsRestClassifier(LogisticRegression(C=C, max_iter=2000, class_weight="balanced")))
])

### Tuning model
In this step, I will use the *general_tuning* function to tune the Logistic Regression and find the threshold that gives the highest micro F1-score on the validation set. The *C_grid* list consists of smaller and larger numbers close to the default value to test different regularization strengths.

In [8]:
import numpy as np

C_grid = [0.25, 0.5, 1.0, 2.0, 4.0]
param_grid = [{"C": C} for C in C_grid]
thresholds = np.arange(0.2, 0.81, 0.05)

best_lr, lr_results = general_tuning(X_train, y_train, X_val, y_val,make_pipeline_fn=make_lr_pipe,param_grid=param_grid,thresholds=thresholds)

print("Best LR params:", {k: best_lr[k] for k in best_lr if k not in ["model"]})
display(lr_results.sort_values("val_micro_f1", ascending=False).head(10))

Best LR params: {'C': 2.0, 'threshold': 0.5499999999999999, 'val_micro_f1': 0.6197391090327344}


,C,threshold,val_micro_f1
3,2.00,0.55,0.619739
4,4.00,0.55,0.616320
2,1.00,0.60,0.614868
1,0.50,0.60,0.608829
0,0.25,0.60,0.592842


The tuning results show that the model scored the highest micro F1-score (0.62) with regularization strength of 2.0 and threshold of nearly 0.55. This suggests that the model performs best on this multi-label classification task with slightly weaker regularization and a threshold just above the default 0.5, which helps balance precision and recall.

### Evaluation
In this step, I will evaluate the performance of the best model using the evaluate function explained in the beginning of the notebook. After that, I will print a classification report to visualise the perfomance of the model on each label separately:

In [9]:
from sklearn.metrics import classification_report

results, y_test_pred = evaluate_model( best_lr["model"], X_test, y_test, best_lr["threshold"], "Logistic Regression")
lr_df = pd.DataFrame([results]).round(2)
lr_df.to_csv("../data/results/logistic_regression_results.csv", index=False)
display(lr_df)
print("Classification report:")
print(classification_report(y_test, y_test_pred, target_names=labels, zero_division=0))

,model,threshold,micro_f1,macro_f1,hamming_loss,jaccard_micro
0,Logistic Regression,0.55,0.61,0.59,0.08,0.44


Classification report:
                     precision    recall  f1-score   support

need_basic_supplies       0.72      0.77      0.75       593
  need_medical_help       0.48      0.56      0.52       416
 need_safety_rescue       0.40      0.47      0.43       243
       need_shelter       0.62      0.71      0.66       344
      people_status       0.51      0.64      0.57       292

          micro avg       0.57      0.65      0.61      1888
          macro avg       0.55      0.63      0.59      1888
       weighted avg       0.58      0.65      0.61      1888
        samples avg       0.21      0.22      0.21      1888



#### Per label Precision/Recall/F1-Score

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

report = classification_report(y_test, y_test_pred,target_names=labels,zero_division=0,output_dict=True)
df_rep = pd.DataFrame(report).T.loc[labels, ["precision", "recall", "f1-score"]]

ax = df_rep.plot(kind="bar", figsize=(10,4))
ax.set_title("Per-label Precision / Recall / F1 (Test set)")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

KeyboardInterrupt: 

#### Multi-label confusion matrices

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import multilabel_confusion_matrix

mcm = multilabel_confusion_matrix(y_test, y_test_pred)
for i, label in enumerate(labels):
    tn, fp, fn, tp = mcm[i].ravel()

    fig, ax = plt.subplots(figsize=(3.2, 3.2))
    ax.imshow([[tn, fp],[fn, tp]])
    ax.set_title(f"Confusion matrix: {label}")
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["Pred 0","Pred 1"])
    ax.set_yticklabels(["True 0","True 1"])

    for (r,c,val) in [(0,0,tn),(0,1,fp),(1,0,fn),(1,1,tp)]:
        ax.text(c, r, str(val), ha="center", va="center")

    plt.tight_layout()
    plt.show()

#### Threshold tuning curve

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

scores_val = get_scores(best_lr["model"], X_val)
thresholds = np.arange(0.2, 0.81, 0.05)
micro_f1s = []
for t in thresholds:
    pred_val = (scores_val >= t).astype(int)
    micro_f1s.append(f1_score(y_val, pred_val, average="micro", zero_division=0))

plt.figure(figsize=(6,4))
plt.plot(thresholds, micro_f1s, marker="o")
plt.title("Validation micro-F1 vs threshold (Logistic Regression)")
plt.xlabel("Threshold")
plt.ylabel("Micro F1")
plt.tight_layout()
plt.show()

## Linear SVC

In [12]:
from sklearn.svm import LinearSVC

def make_svc_pipe(C):
    return Pipeline([
        ("tfidf", make_tfidf()),
        ("clf", OneVsRestClassifier(LinearSVC(C=C, class_weight="balanced", max_iter=5000)))
    ])

#### Tuning model

In [13]:
svm_probe = make_svc_pipe(C=1.0)
svm_probe.fit(X_train, y_train)

val_scores = get_scores(svm_probe, X_val)
thresholds_svm = np.percentile(val_scores, np.linspace(5, 95, 25))
C_grid = [0.25, 0.5, 1.0, 2.0, 4.0]
param_grid = [{"C": C} for C in C_grid]

best_svm, svm_results = general_tuning(
    X_train, y_train, X_val, y_val,
    make_pipeline_fn=make_svc_pipe,
    param_grid=param_grid,
    thresholds=thresholds_svm
)

print("Best SVC params:", {k: best_svm[k] for k in best_svm if k not in ["model"]})
display(svm_results.sort_values("val_micro_f1", ascending=False).head(10))

Best SVC params: {'C': 0.25, 'threshold': 0.0779554154008746, 'val_micro_f1': 0.6190114068441065}


,C,threshold,val_micro_f1
0,0.25,0.077955,0.619011
1,0.50,0.077955,0.608097
2,1.00,0.077955,0.595395
3,2.00,-0.163048,0.585752
4,4.00,-0.163048,0.579689


#### Evaluation
In this step, I will evaluate the performance of the best model using the evaluate function explained in the beginning of the notebook. After that, I will print a classification report to visualise the perfomance of the model on each label separately:

In [14]:
from sklearn.metrics import classification_report

results, y_test_pred = evaluate_model(best_svm["model"], X_test, y_test, best_svm["threshold"],"LinearSVC")

svm_df = pd.DataFrame([results]).round(2)
svm_df.to_csv("../data/results/linear_svc_results.csv", index=False)

display(svm_df)
print("Classification report:")
print(classification_report(y_test, y_test_pred, target_names=labels, zero_division=0))

,model,threshold,micro_f1,macro_f1,hamming_loss,jaccard_micro
0,LinearSVC,0.08,0.61,0.58,0.08,0.44


Classification report:
                     precision    recall  f1-score   support

need_basic_supplies       0.74      0.77      0.76       593
  need_medical_help       0.49      0.54      0.52       416
 need_safety_rescue       0.43      0.44      0.43       243
       need_shelter       0.63      0.67      0.65       344
      people_status       0.52      0.57      0.54       292

          micro avg       0.59      0.63      0.61      1888
          macro avg       0.56      0.60      0.58      1888
       weighted avg       0.59      0.63      0.61      1888
        samples avg       0.21      0.22      0.21      1888



# Model comparison